# Predicting arrears next month
Protocol: `protocol.yaml`. Temporal holdout from `holdout.csv`, 5-split walk-forward CV on training rows, selection by mean CV ROC AUC.

**Feature decision:** `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time (86.5% arrears when 1 vs 1.4% when 0). It is excluded as leakage. `meter_ref` and `billing_month` are excluded per protocol.

In [ ]:
import json, hashlib, numpy as np, pandas as pd, yaml
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

proto = yaml.safe_load(open("protocol.yaml"))
assert hashlib.sha256(open("data.csv","rb").read()).hexdigest() == proto["dataset"]["sha256"]
assert hashlib.sha256(open("holdout.csv","rb").read()).hexdigest() == proto["split"]["holdout_sha256"]
df = pd.read_csv("data.csv")
assert len(df) == proto["dataset"]["rows"]
hold_idx = pd.read_csv("holdout.csv")["row"].to_numpy()
assert len(hold_idx) == proto["split"]["holdout_rows"]
is_hold = df.index.isin(hold_idx)
train, hold = df[~is_hold], df[is_hold]
TARGET = "arrears_next_month"
CAT = ["tariff"]
NUM = ["occupants", "kwh", "bill_eur", "late_payments_12m"]
FEATURES = CAT + NUM   # final_notice_sent excluded: recorded after the outcome
print(len(train), len(hold), train[TARGET].mean(), hold[TARGET].mean())

## Candidates (named before any scoring)

In [ ]:
def prep(scale):
    return ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), CAT),
                              ("n", StandardScaler() if scale else "passthrough", NUM)])
candidates = {
    "baseline_prior": Pipeline([("p", prep(False)), ("m", DummyClassifier(strategy="prior"))]),
    "logreg_C0.1": Pipeline([("p", prep(True)), ("m", LogisticRegression(C=0.1, max_iter=1000))]),
    "logreg_C1": Pipeline([("p", prep(True)), ("m", LogisticRegression(C=1.0, max_iter=1000))]),
    "hgb_d3": Pipeline([("p", prep(False)), ("m", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))]),
}
print(list(candidates))

## Walk-forward CV on training rows (expanding window over billing months)

In [ ]:
months = np.sort(train["billing_month"].unique())
n_splits = proto["cv"]["n_splits"]
blocks = np.array_split(months, n_splits + 1)
folds = []
for k in range(1, n_splits + 1):
    tr_m = np.concatenate(blocks[:k]); va_m = blocks[k]
    folds.append((np.where(train.billing_month.isin(tr_m))[0], np.where(train.billing_month.isin(va_m))[0]))
cv_scores = {}
for name, est in candidates.items():
    s = []
    for tr, va in folds:
        m = clone(est).fit(train.iloc[tr][FEATURES], train.iloc[tr][TARGET])   # preprocessing fitted inside fold
        s.append(roc_auc_score(train.iloc[va][TARGET], m.predict_proba(train.iloc[va][FEATURES])[:, 1]))
    cv_scores[name] = s
pd.DataFrame(cv_scores).T.assign(mean=lambda d: d.mean(axis=1))

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print("chosen:", chosen)
final = clone(candidates[chosen]).fit(train[FEATURES], train[TARGET])
holdout_auc = roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:, 1])   # scored once
print("holdout roc_auc:", holdout_auc)

In [ ]:
pos = train.index.to_numpy()
evidence = {
    "features": FEATURES,
    "candidates": list(candidates),
    "cv_folds": [[pos[tr].tolist(), pos[va].tolist()] for tr, va in folds],
    "cv_scores": {k: [float(x) for x in v] for k, v in cv_scores.items()},
    "chosen": chosen,
    "holdout_rows": hold.index.tolist(),
    "holdout_scores": {chosen: float(holdout_auc)},
}
json.dump(evidence, open("evidence.json", "w"))

## Conclusion
See the CV table and the holdout ROC AUC above (chosen candidate, refit on all training rows, scored once). The strongest apparent signal, `final_notice_sent`, was deliberately not used because it is not available at prediction time.